# Import Libraries

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Load Dataset

The Lending Club dataset is loaded from the specified local directory.

A backup copy of the raw dataset is preserved before any transformation.  

In [2]:
# =============================================================================
# Load dataset
# =============================================================================

DATA_PATH = Path("../data/raw/loan_data_2007_2014.csv")

assert DATA_PATH.exists(), f"Dataset not found: {DATA_PATH}"

loan_data_backup = pd.read_csv(DATA_PATH)


C:\Users\Platini AGOUANET\AppData\Local\Temp\ipykernel_24468\3877402495.py:9: DtypeWarning: Columns (19) have mixed types. Specify dtype option on import or set low_memory=False.
  loan_data_backup = pd.read_csv(DATA_PATH)


In [3]:
loan_data=loan_data_backup.copy()
loan_data=loan_data.drop(columns=loan_data.columns[0])

# Initial Data Exploration

In [4]:
loan_data.head()

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,total_bal_il,il_util,open_rv_12m,open_rv_24m,max_bal_bc,all_util,total_rev_hi_lim,inq_fi,total_cu_tl,inq_last_12m
0,1077501,1296599,5000,5000,4975.0,36 months,10.65,162.87,B,B2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1077430,1314167,2500,2500,2500.0,60 months,15.27,59.83,C,C4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1077175,1313524,2400,2400,2400.0,36 months,15.96,84.33,C,C5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1076863,1277178,10000,10000,10000.0,36 months,13.49,339.31,C,C1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1075358,1311748,3000,3000,3000.0,60 months,12.69,67.79,B,B5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
print(f"Number of observations : {loan_data.shape[0]:,}")
print(f"Number of features     : {loan_data.shape[1]}")

Number of observations : 466,285
Number of features     : 73


In [6]:
#Duplicate rows
duplicates = loan_data.duplicated().sum()

print(f"Duplicate rows: {duplicates:,}")


Duplicate rows: 0


In [7]:
# Assess missing values and remove fully empty variables

missing_summary = (
    loan_data.isna()
    .agg(["sum", "mean"])
    .T
    .rename(columns={
        "sum": "Missing Values",
        "mean": "Missing Percentage"
    })
)

missing_summary["Missing Values"] = (
    missing_summary["Missing Values"].astype(int)
)

missing_summary["Missing Percentage"] = (
    missing_summary["Missing Percentage"] * 100
).round(2)

missing_summary = (
    missing_summary[missing_summary["Missing Values"] > 0]
    .sort_values("Missing Percentage", ascending=False)
)

display(missing_summary)

# Identify and remove variables containing 100% missing values
empty_columns = missing_summary.index[
    missing_summary["Missing Percentage"].eq(100)
].tolist()

initial_shape = loan_data.shape

if empty_columns:
    loan_data = loan_data.drop(columns=empty_columns)

print(f"Initial dataset shape: {initial_shape}")
print(f"Variables removed: {len(empty_columns)}")
print(f"Removed variables: {empty_columns}")
print(f"Final dataset shape: {loan_data.shape}")

,Missing Values,Missing Percentage
inq_fi,466285,100.00
open_rv_24m,466285,100.00
max_bal_bc,466285,100.00
all_util,466285,100.00
inq_last_12m,466285,100.00
annual_inc_joint,466285,100.00
verification_status_joint,466285,100.00
dti_joint,466285,100.00
total_cu_tl,466285,100.00
il_util,466285,100.00


Initial dataset shape: (466285, 73)
Variables removed: 17
Removed variables: ['inq_fi', 'open_rv_24m', 'max_bal_bc', 'all_util', 'inq_last_12m', 'annual_inc_joint', 'verification_status_joint', 'dti_joint', 'total_cu_tl', 'il_util', 'mths_since_rcnt_il', 'total_bal_il', 'open_il_24m', 'open_il_12m', 'open_il_6m', 'open_acc_6m', 'open_rv_12m']
Final dataset shape: (466285, 56)


In [8]:
#Data types
loan_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 466285 entries, 0 to 466284
Data columns (total 56 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   id                           466285 non-null  int64  
 1   member_id                    466285 non-null  int64  
 2   loan_amnt                    466285 non-null  int64  
 3   funded_amnt                  466285 non-null  int64  
 4   funded_amnt_inv              466285 non-null  float64
 5   term                         466285 non-null  object 
 6   int_rate                     466285 non-null  float64
 7   installment                  466285 non-null  float64
 8   grade                        466285 non-null  object 
 9   sub_grade                    466285 non-null  object 
 10  emp_title                    438697 non-null  object 
 11  emp_length                   445277 non-null  object 
 12  home_ownership               466285 non-null  object 
 13 

# Data Type Assessment and Conversion

The objective of this section is to verify whether the data types assigned by pandas correctly represent the business meaning of each variable.

An inappropriate data type can lead to inefficient storage, incorrect analyses, or difficulties during feature engineering and model development.

For each variable requiring attention, the following steps will be performed:

1. Understand the business meaning of the variable.
2. Compare the business type with the pandas data type.
3. Decide whether a transformation is required.
4. Apply the transformation if necessary.
5. Validate the result.

## Variable: `term`

The variable `term` represents the contractual duration of the loan, expressed in months.

From a business perspective, this variable is numerical because it measures a duration. However, it is currently stored as an `object` because its values contain text such as `"36 months"` and `"60 months"`.

The text suffix does not provide additional information for modeling. Therefore, the numerical component will be extracted and the variable will be converted from `object` to `int16`.

**Current type:** `object`  
**Target type:** `int16`  
**Decision:** Transform and keep.

In [9]:
#remove months in term column and convert to int
#loan_data['term'] = loan_data['term'].str.replace('months', '').astype(int)
#print(loan_data['term'].head())

In [10]:
# Transform the loan term into a numerical duration in months
loan_data["term"] = (
    loan_data["term"]
    .str.extract(r"(\d+)", expand=False)
    .astype("int16")
)

# Validate the transformation
display(pd.DataFrame({
    "Data Type": [loan_data["term"].dtype],
    "Missing Values": [loan_data["term"].isna().sum()],
    "Unique Values": [loan_data["term"].nunique()]
}))

loan_data["term"].value_counts(dropna=False).sort_index()

,Data Type,Missing Values,Unique Values
0,int16,0,2


term
36    337953
60    128332
Name: count, dtype: int64

## Date Variables

The following variables represent dates but are currently stored as `object`:

- `issue_d`
- `earliest_cr_line`
- `last_pymnt_d`
- `next_pymnt_d`
- `last_credit_pull_d`

Although some variables already contain missing values, these missing values will be preserved during the conversion process.

To ensure proper chronological analysis and feature engineering, all date variables will be converted from `object` to `datetime64[ns]`.

Missing or invalid values will be represented as `NaT` (Not a Time).

**Decision:** Convert all date variables while preserving missing values.

In [11]:
# Date variables
date_columns = [
    "issue_d",
    "earliest_cr_line",
    "last_pymnt_d",
    "next_pymnt_d",
    "last_credit_pull_d"
]

# Convert all date variables
for column in date_columns:
    loan_data[column] = pd.to_datetime(
        loan_data[column],
        format="%b-%y",
        errors="coerce"
    )
    
# Validate the transformations
date_validation = pd.DataFrame({
    "data_type": loan_data[date_columns].dtypes.astype(str),
    "missing_values": loan_data[date_columns].isna().sum(),
    "minimum_date": loan_data[date_columns].min(),
    "maximum_date": loan_data[date_columns].max()
})

date_validation

KeyError: 'issue_d'

In [ ]:
print(loan_data['issue_d'].min())
print(loan_data['issue_d'].max())

2007-06-01 00:00:00
2014-12-01 00:00:00


**Business Validation: `earliest_cr_line`**

During the validation step, some values of `earliest_cr_line` were found to occur after the loan issuance period.

This issue results from the default interpretation of two-digit years by pandas during the conversion to `datetime`.

Since `earliest_cr_line` represents the borrower's first credit line, it cannot logically occur after the loan was issued.

Therefore, dates interpreted in the future were corrected by shifting them back by 100 years, restoring their intended historical values.

In [ ]:
latest_issue_year = loan_data["issue_d"].dt.year.max()

mask = loan_data["earliest_cr_line"].dt.year > latest_issue_year

loan_data.loc[mask, "earliest_cr_line"] = (
    loan_data.loc[mask, "earliest_cr_line"]
    - pd.DateOffset(years=100)
)

In [ ]:
validation = pd.DataFrame({
    "minimum_date": loan_data[date_columns].min(),
    "maximum_date": loan_data[date_columns].max()
})

display(validation)

,minimum_date,maximum_date
issue_d,2007-06-01,2014-12-01
earliest_cr_line,1944-01-01,2011-11-01
last_pymnt_d,2007-12-01,2016-01-01
next_pymnt_d,2007-12-01,2016-03-01
last_credit_pull_d,2007-05-01,2016-01-01


# Categorical Variables Assessment

This section examines the categorical variables available at the time of the credit decision.

The objective is not yet to encode these variables, but rather to understand their business meaning, evaluate their data quality, and determine the appropriate preprocessing strategy for subsequent modeling.

For each variable (or group of related variables), we will:

1. Understand its business meaning.
2. Examine its categories.
3. Assess whether it contains useful predictive information.
4. Decide whether to keep, transform, or remove it.

## Loan Quality: `grade` and `sub_grade`

The variables `grade` and `sub_grade` represent Lending Club's internal assessment of the borrower's credit risk.

`grade` provides a broad credit risk classification ranging from A (lowest risk) to G (highest risk), while `sub_grade` offers a more detailed assessment within each grade (e.g., A1, A2, B3).

Both variables are ordinal categorical variables and are available at the time the loan application is evaluated.

Since `sub_grade` appears to be a more granular version of `grade`, the two variables may contain redundant information.

The objective of this assessment is to:

- examine the distribution of both variables;
- verify the consistency between `grade` and `sub_grade`;
- determine whether both variables are required for modeling.

**Current type:** `object`

**Business type:** Ordinal categorical

**Preliminary decision:** Assess whether `sub_grade` completely determines `grade`. If confirmed, retain only `sub_grade` for modeling to avoid redundant information.

In [ ]:
# Analyze and validate the relationship between grade and sub_grade

grade_validation = pd.DataFrame({
    "Missing Values": loan_data[["grade", "sub_grade"]].isna().sum(),
    "Unique Values": loan_data[["grade", "sub_grade"]].nunique()
})

# Check consistency only for records where both variables are available
complete_grade_records = (
    loan_data["grade"].notna()
    & loan_data["sub_grade"].notna()
)

grade_consistency = (
    loan_data.loc[complete_grade_records, "sub_grade"]
    .astype("string")
    .str[0]
    ==
    loan_data.loc[complete_grade_records, "grade"]
    .astype("string")
)

# Create the grade/sub-grade distribution table
grades = list("ABCDEFG")

grade_subgrade_distribution = pd.DataFrame(
    index=grades,
    columns=[
        "Grade Total",
        "1",
        "2",
        "3",
        "4",
        "5",
        "Sub-grade Total",
        "Difference",
        "Match"
    ]
)

for grade in grades:

    grade_total = loan_data["grade"].eq(grade).sum()

    subgrade_counts = [
        loan_data["sub_grade"].eq(f"{grade}{level}").sum()
        for level in range(1, 6)
    ]

    subgrade_total = sum(subgrade_counts)

    grade_subgrade_distribution.loc[grade] = [
        grade_total,
        *subgrade_counts,
        subgrade_total,
        grade_total - subgrade_total,
        grade_total == subgrade_total
    ]

# Add missing values as the last row
missing_grade = loan_data["grade"].isna().sum()
missing_subgrade = loan_data["sub_grade"].isna().sum()

grade_subgrade_distribution.loc["Missing"] = [
    missing_grade,
    pd.NA,
    pd.NA,
    pd.NA,
    pd.NA,
    pd.NA,
    missing_subgrade,
    missing_grade - missing_subgrade,
    missing_grade == missing_subgrade
]

# Convert count columns to nullable integers
count_columns = [
    "Grade Total",
    "1",
    "2",
    "3",
    "4",
    "5",
    "Sub-grade Total",
    "Difference"
]

grade_subgrade_distribution[count_columns] = (
    grade_subgrade_distribution[count_columns]
    .astype("Int64")
)

print("Variable summary:")
display(grade_validation)

print("\nGrade and sub-grade distribution:")
display(grade_subgrade_distribution)

print(
    f"\nInconsistent grade/sub_grade records: "
    f"{(~grade_consistency).sum():,}"
)

Variable summary:


,Missing Values,Unique Values
grade,0,7
sub_grade,0,35



Grade and sub-grade distribution:


,Grade Total,1,2,3,4,5,Sub-grade Total,Difference,Match
A,74867,10541,10956,12568,19045,21757,74867,0,True
B,136929,22876,26610,31686,30505,25252,136929,0,True
C,125293,26953,26740,25317,24105,22178,125293,0,True
D,76888,19261,17046,14916,14099,11566,76888,0,True
E,35757,9033,8669,6976,5992,5087,35757,0,True
F,13229,3940,3001,2708,2067,1513,13229,0,True
G,3322,1109,823,583,422,385,3322,0,True
Missing,0,<NA>,<NA>,<NA>,<NA>,<NA>,0,0,True



Inconsistent grade/sub_grade records: 0


**Business Assessment**

The analysis shows that both variables are complete, with no missing values.

The variable `grade` contains seven ordered credit risk categories (A to G), while `sub_grade` provides a finer classification with 35 ordered categories (A1 to G5).

The consistency check confirms that every value of `sub_grade` corresponds to its associated `grade`, indicating that `sub_grade` completely determines `grade`.

**Decision**

Since `sub_grade` contains all the information provided by `grade` at a higher level of granularity, keeping both variables would introduce redundant information into the model.

Therefore, `sub_grade` will be retained for modeling, while `grade` will be excluded from the final modeling dataset.

## Employment Information: `emp_length` and `emp_title`

The variables `emp_length` and `emp_title` describe the borrower's employment status at the time of the loan application.

`emp_length` indicates the length of the borrower's employment, while `emp_title` contains the reported job title.

These variables may provide useful information about the borrower's financial stability. However, they differ considerably in structure: `emp_length` is an ordinal categorical variable with a limited number of categories, whereas `emp_title` is a free-text variable that may contain many unique values, spelling variations, and inconsistencies.

The objective of this assessment is to:

- examine the completeness of both variables;
- evaluate the structure of their categories;
- identify potential data quality issues;
- determine the appropriate preprocessing strategy for each variable.

**Current type:** `object`

**Business type:**
- `emp_length`: Ordinal categorical
- `emp_title`: Free-text categorical

**Preliminary decision:** Assess whether `emp_length` can be standardized and whether `emp_title` contains sufficiently consistent information to be useful for predictive modeling.

In [ ]:
# Employment information assessment

employment_summary = pd.DataFrame({
    "missing_values": loan_data[["emp_length", "emp_title"]].isna().sum(),
    "unique_values": loan_data[["emp_length", "emp_title"]].nunique()
})

print("Variable summary:")
display(employment_summary)

print("\nEmployment length distribution:")
display(loan_data["emp_length"].value_counts(dropna=False).sort_index())

print("\nTop 20 employment titles:")
display(loan_data["emp_title"].value_counts(dropna=False).head(20))

# Preliminary decision
print("\nDecision:")
print("- Review and standardize 'emp_length' if necessary.")
print("- Evaluate whether 'emp_title' should be retained, grouped into broader categories, or excluded due to its high cardinality.")

Variable summary:


,missing_values,unique_values
emp_length,21008,11
emp_title,27588,205475



Employment length distribution:


emp_length
1 year        29622
10+ years    150049
2 years       41373
3 years       36596
4 years       28023
5 years       30774
6 years       26112
7 years       26180
8 years       22395
9 years       17888
< 1 year      36265
NaN           21008
Name: count, dtype: int64


Top 20 employment titles:


emp_title
NaN                 27588
Teacher              5399
Manager              4438
Registered Nurse     2316
RN                   2204
Supervisor           1967
Project Manager      1624
Sales                1624
Owner                1527
Office Manager       1395
manager              1312
Driver               1296
General Manager      1263
Director             1187
teacher              1182
Engineer             1049
driver                967
Vice President        944
President             915
owner                 856
Name: count, dtype: int64


Decision:
- Review and standardize 'emp_length' if necessary.
- Evaluate whether 'emp_title' should be retained, grouped into broader categories, or excluded due to its high cardinality.


**Business Assessment**

The analysis shows that `emp_length` contains only 11 ordered categories and a relatively small proportion of missing values (about 4.5%). Since employment duration is a proxy for job stability, this variable is expected to provide useful predictive information for credit risk assessment. However, its values are currently stored as text and will require standardization before modeling.

In contrast, `emp_title` contains more than 205,000 unique values, indicating extremely high cardinality. The most frequent job titles also reveal inconsistencies in capitalization and spelling (e.g., "Manager" and "manager", "Teacher" and "teacher", "Owner" and "owner"), suggesting that the variable is not standardized. Without substantial text preprocessing and grouping into broader occupational categories, its direct use in a predictive model would likely introduce noise rather than useful information.

**Decision**

`emp_length` will be retained and standardized during feature engineering to preserve its ordinal information.

`emp_title` will not be used in its current form. It will either require extensive text preprocessing and categorization or be excluded from the final modeling dataset if no meaningful transformation is performed.

## Home Ownership: `home_ownership`

The variable `home_ownership` describes the borrower's housing status at the time of the loan application.

Home ownership may reflect the borrower's financial stability and therefore could be associated with credit risk.

The objective of this assessment is to examine the distribution of the different housing categories and identify any data quality issues.

**Current type:** `object`

**Business type:** Nominal categorical

In [ ]:
# Home ownership assessment

home_ownership_summary = pd.DataFrame({
    "missing_values": loan_data[["home_ownership"]].isna().sum(),
    "unique_values": loan_data[["home_ownership"]].nunique()
})

print("Variable summary:")
display(home_ownership_summary)

print("\nHome ownership distribution:")
display(loan_data["home_ownership"].value_counts(dropna=False))

Variable summary:


,missing_values,unique_values
home_ownership,0,6



Home ownership distribution:


home_ownership
MORTGAGE    235875
RENT        188473
OWN          41704
OTHER          182
NONE            50
ANY              1
Name: count, dtype: int64

## Income Verification: `verification_status`

The variable `verification_status` indicates whether the borrower's reported income was verified during the loan application process.

Income verification may improve the reliability of the borrower's financial information and therefore could influence credit risk assessment.

The objective of this assessment is to examine the distribution of the verification categories and identify any potential data quality issues.

**Current type:** `object`

**Business type:** Nominal categorical

In [ ]:
# Income verification assessment

verification_summary = pd.DataFrame({
    "missing_values": loan_data[["verification_status"]].isna().sum(),
    "unique_values": loan_data[["verification_status"]].nunique()
})

print("Variable summary:")
display(verification_summary)

print("\nVerification status distribution:")
display(loan_data["verification_status"].value_counts(dropna=False))

Variable summary:


,missing_values,unique_values
verification_status,0,3



Verification status distribution:


verification_status
Verified           168055
Source Verified    149993
Not Verified       148237
Name: count, dtype: int64

## Loan Purpose: `purpose` and `title`

The variables `purpose` and `title` describe the reason for which the borrower requested the loan.

`purpose` is a standardized categorical variable defined by Lending Club, whereas `title` contains the borrower's own description of the loan purpose.

The objective of this assessment is to examine the quality of both variables, compare their level of detail, and identify any potential data quality issues.

**Current type:** `object`

**Business type:**
- `purpose`: Nominal categorical
- `title`: Free-text categorical

In [ ]:
# Loan purpose assessment

purpose_summary = pd.DataFrame({
    "missing_values": loan_data[["purpose", "title"]].isna().sum(),
    "unique_values": loan_data[["purpose", "title"]].nunique()
})

print("Variable summary:")
display(purpose_summary)

print("\nPurpose distribution:")
display(loan_data["purpose"].value_counts(dropna=False))


Variable summary:


,missing_values,unique_values
purpose,0,14
title,21,63098



Purpose distribution:


purpose
debt_consolidation    274195
credit_card           104157
home_improvement       26537
other                  23690
major_purchase          9828
small_business          7013
car                     5397
medical                 4602
moving                  2994
vacation                2487
wedding                 2343
house                   2269
educational              422
renewable_energy         351
Name: count, dtype: int64

## Geographic Information: `addr_state` and `zip_code`

The variables `addr_state` and `zip_code` provide information about the borrower's location.

Geographic information may capture regional economic conditions and lending patterns that could influence credit risk.

The objective of this assessment is to examine the completeness and distribution of both variables and identify any potential data quality issues.

**Current type:** `object`

**Business type:**
- `addr_state`: Nominal categorical
- `zip_code`: Nominal categorical

In [ ]:
# Geographic information assessment

geographic_summary = pd.DataFrame({
    "missing_values": loan_data[["addr_state", "zip_code"]].isna().sum(),
    "unique_values": loan_data[["addr_state", "zip_code"]].nunique()
})

print("Variable summary:")
display(geographic_summary)

print("\nState distribution:")
display(loan_data["addr_state"].value_counts(dropna=False))

print("\nTop ZIP code prefixes:")
display(loan_data["zip_code"].value_counts(dropna=False).head(20))

Variable summary:


,missing_values,unique_values
addr_state,0,50
zip_code,0,888



State distribution:


addr_state
CA    71450
NY    40242
TX    36439
FL    31637
IL    18612
NJ    18061
PA    16424
OH    15237
GA    14975
VA    14222
NC    12682
MI    11549
MA    11072
MD    10974
AZ    10712
WA    10517
CO     9739
MN     8158
MO     7508
CT     7204
IN     6525
NV     6519
TN     5984
OR     5949
WI     5911
AL     5853
SC     5583
LA     5489
KY     4438
KS     4190
OK     4117
AR     3488
UT     3428
NM     2588
HI     2487
WV     2412
NH     2232
RI     2050
DC     1426
MT     1396
DE     1272
AK     1251
MS     1226
WY     1128
SD      980
VT      905
IA       14
NE       14
ID       12
ME        4
Name: count, dtype: int64


Top ZIP code prefixes:


zip_code
945xx    5304
112xx    5102
750xx    5013
606xx    4696
100xx    4391
300xx    4120
900xx    4069
070xx    4067
331xx    3983
770xx    3587
917xx    3552
891xx    3338
330xx    3254
104xx    3237
117xx    3231
921xx    3152
926xx    3001
852xx    2929
913xx    2859
113xx    2684
Name: count, dtype: int64

## Lending Club Information: `application_type`, `initial_list_status` and `pymnt_plan`

The variables `application_type`, `initial_list_status`, and `pymnt_plan` describe administrative characteristics of the loan application process.

These variables may provide additional information about how the loan was originated and managed by Lending Club.

The objective of this assessment is to examine the completeness and distribution of these variables and identify any potential data quality issues.

**Current type:** `object`

**Business type:** Nominal categorical

In [ ]:
# Lending Club information assessment

lending_summary = pd.DataFrame({
    "missing_values": loan_data[
        ["application_type", "initial_list_status", "pymnt_plan"]
    ].isna().sum(),
    "unique_values": loan_data[
        ["application_type", "initial_list_status", "pymnt_plan"]
    ].nunique()
})

print("Variable summary:")
display(lending_summary)

for column in ["application_type", "initial_list_status", "pymnt_plan"]:
    print(f"\n{column} distribution:")
    display(loan_data[column].value_counts(dropna=False))

Variable summary:


,missing_values,unique_values
application_type,0,1
initial_list_status,0,2
pymnt_plan,0,2



application_type distribution:


application_type
INDIVIDUAL    466285
Name: count, dtype: int64


initial_list_status distribution:


initial_list_status
f    303005
w    163280
Name: count, dtype: int64


pymnt_plan distribution:


pymnt_plan
n    466276
y         9
Name: count, dtype: int64

**Decision on `application_type`, `initial_list_status`, and `pymnt_plan`**

These three variables describe characteristics of the loan application and listing process. Their distributions were examined to assess whether they contain sufficient variability to contribute meaningful predictive information.

The `application_type` variable contains only a single category (`INDIVIDUAL`) across all observations. As a constant feature, it provides no discriminatory information and will therefore be removed from the master dataset.

The `initial_list_status` variable contains two well-represented categories (`f` and `w`), indicating sufficient variability to potentially capture information related to the loan listing process. Since no evidence currently supports its removal, it will be retained for subsequent feature engineering and predictive modeling.

The `pymnt_plan` variable is extremely imbalanced, with only **9 observations** belonging to the minority category (`y`). Given its near-zero variance, the variable is unlikely to provide meaningful predictive information and will therefore be removed from the master dataset.

## Target Variable: `loan_status`

The variable `loan_status` represents the final status of each loan and serves as the target variable for credit risk modeling.

It identifies whether a loan was successfully repaid, is currently active, or experienced repayment difficulties.

The objective of this assessment is to examine the distribution of the different loan status categories before defining the target variable used for modeling.

**Current type:** `object`

**Business type:** Nominal categorical (Target variable)

In [ ]:
# Target variable assessment

target_summary = pd.DataFrame({
    "missing_values": loan_data[["loan_status"]].isna().sum(),
    "unique_values": loan_data[["loan_status"]].nunique()
})

print("Variable summary:")
display(target_summary)

print("\nLoan status distribution:")
display(loan_data["loan_status"].value_counts(dropna=False))

Variable summary:


,missing_values,unique_values
loan_status,0,9



Loan status distribution:


loan_status
Current                                                224226
Fully Paid                                             184739
Charged Off                                             42475
Late (31-120 days)                                       6900
In Grace Period                                          3146
Does not meet the credit policy. Status:Fully Paid       1988
Late (16-30 days)                                        1218
Default                                                   832
Does not meet the credit policy. Status:Charged Off       761
Name: count, dtype: int64

# Numerical Variables Assessment

## Loan Characteristics

The variables in this group describe the financial characteristics of the loan at the time of origination.

They include the requested loan amount, the funded amount, the investor-funded amount, the loan term, the interest rate, and the monthly installment.

The objective of this assessment is to examine the completeness, distribution, and basic descriptive statistics of these variables before any preprocessing.

**Business type:** Numerical

In [ ]:
# Loan characteristics assessment

loan_characteristics = [
    "loan_amnt",
    "funded_amnt",
    "funded_amnt_inv",
    "term",
    "int_rate",
    "installment"
]

summary = pd.DataFrame({
    "missing_values": loan_data[loan_characteristics].isna().sum(),
    "data_type": loan_data[loan_characteristics].dtypes
})

print("Variable summary:")
display(summary)

print("\nDescriptive statistics:")
display(loan_data[loan_characteristics].describe().T)

Variable summary:


,missing_values,data_type
loan_amnt,0,int64
funded_amnt,0,int64
funded_amnt_inv,0,float64
term,0,int16
int_rate,0,float64
installment,0,float64



Descriptive statistics:


,count,mean,std,min,25%,50%,75%,max
loan_amnt,466285.0,14317.277577,8286.509164,500.00,8000.00,12000.00,20000.00,35000.00
funded_amnt,466285.0,14291.801044,8274.371300,500.00,8000.00,12000.00,20000.00,35000.00
funded_amnt_inv,466285.0,14222.329888,8297.637788,0.00,8000.00,12000.00,19950.00,35000.00
term,466285.0,42.605334,10.719040,36.00,36.00,36.00,60.00,60.00
int_rate,466285.0,13.829236,4.357587,5.42,10.99,13.66,16.49,26.06
installment,466285.0,432.061201,243.485550,15.67,256.69,379.89,566.58,1409.99


**Decision on Loan Amount Variables**

The variables `loan_amnt`, `funded_amnt`, and `funded_amnt_inv` describe different aspects of the loan financing process and provide complementary financial information. Although these variables are expected to be highly correlated, each captures a distinct business concept.

At this stage, all three variables will be retained in the master dataset. Potential redundancy and multicollinearity will be evaluated during feature selection and model development.

**Decision:** Retain all loan amount variables for subsequent analysis.

In [ ]:
#linear correlation matrix for numerical variables
print(loan_data[loan_characteristics].corr())

                 loan_amnt  funded_amnt  funded_amnt_inv      term  int_rate  \
loan_amnt         1.000000     0.998548         0.994347  0.412870  0.167183   
funded_amnt       0.998548     1.000000         0.996125  0.410862  0.167921   
funded_amnt_inv   0.994347     0.996125         1.000000  0.412005  0.169353   
term              0.412870     0.410862         0.412005  1.000000  0.443395   
int_rate          0.167183     0.167921         0.169353  0.443395  1.000000   
installment       0.949666     0.951787         0.947387  0.159631  0.148790   

                 installment  
loan_amnt           0.949666  
funded_amnt         0.951787  
funded_amnt_inv     0.947387  
term                0.159631  
int_rate            0.148790  
installment         1.000000  


## Borrower Financial Information

The variables `annual_inc` and `dti` describe the borrower's financial capacity at the time of the loan application.

Annual income reflects the borrower's reported earnings, while the debt-to-income ratio measures the proportion of monthly debt obligations relative to income.

The objective of this assessment is to examine the completeness and descriptive statistics of these variables.

**Business type:** Numerical

In [ ]:
# Borrower financial information assessment

borrower_financial = [
    "annual_inc",
    "dti"
]

borrower_financial_summary = pd.DataFrame({
    "data_type": loan_data[borrower_financial].dtypes.astype(str),
    "missing_values": loan_data[borrower_financial].isna().sum()
})

print("Variable summary:")
display(borrower_financial_summary)

print("\nDescriptive statistics:")
display(loan_data[borrower_financial].describe().T)

Variable summary:


,data_type,missing_values
annual_inc,float64,4
dti,float64,0



Descriptive statistics:


,count,mean,std,min,25%,50%,75%,max
annual_inc,466281.0,73277.381470,54963.568654,1896.0,45000.00,63000.00,88960.00,7500000.00
dti,466285.0,17.218758,7.851121,0.0,11.36,16.87,22.78,39.99


## Credit History

These variables describe the borrower's historical credit behavior before the loan was issued.

They include information about previous delinquencies, credit inquiries, open accounts, public records, total credit accounts, and current delinquencies.

The objective of this assessment is to examine the completeness and descriptive statistics of these variables.

**Business type:** Numerical

In [ ]:
# Credit history assessment

credit_history = [
    "delinq_2yrs",
    "inq_last_6mths",
    "open_acc",
    "pub_rec",
    "total_acc",
    "acc_now_delinq"
]

credit_history_summary = pd.DataFrame({
    "data_type": loan_data[credit_history].dtypes.astype(str),
    "missing_values": loan_data[credit_history].isna().sum()
})

print("Variable summary:")
display(credit_history_summary)

print("\nDescriptive statistics:")
display(loan_data[credit_history].describe().T)

Variable summary:


,data_type,missing_values
delinq_2yrs,float64,29
inq_last_6mths,float64,29
open_acc,float64,29
pub_rec,float64,29
total_acc,float64,29
acc_now_delinq,float64,29



Descriptive statistics:


,count,mean,std,min,25%,50%,75%,max
delinq_2yrs,466256.0,0.284678,0.797365,0.0,0.0,0.0,0.0,29.0
inq_last_6mths,466256.0,0.804745,1.091598,0.0,0.0,0.0,1.0,33.0
open_acc,466256.0,11.187069,4.987526,0.0,8.0,10.0,14.0,84.0
pub_rec,466256.0,0.160564,0.510863,0.0,0.0,0.0,0.0,63.0
total_acc,466256.0,25.064430,11.600141,1.0,17.0,23.0,32.0,156.0
acc_now_delinq,466256.0,0.004002,0.068637,0.0,0.0,0.0,0.0,5.0


## Revolving Credit

The variables in this group describe the borrower's revolving credit profile at the time of the loan application.

They include the outstanding revolving balance, revolving credit utilization, total revolving credit limit, current total balance across accounts, and total amount ever sent to collections.

The objective of this assessment is to examine the completeness and descriptive statistics of these variables before feature engineering.

**Business type:** Numerical

In [ ]:
# Revolving credit assessment

revolving_credit = [
    "revol_bal",
    "revol_util",
    "total_rev_hi_lim",
    "tot_cur_bal",
    "tot_coll_amt"
]

revolving_credit_summary = pd.DataFrame({
    "data_type": loan_data[revolving_credit].dtypes.astype(str),
    "missing_values": loan_data[revolving_credit].isna().sum()
})

print("Variable summary:")
display(revolving_credit_summary)

print("\nDescriptive statistics:")
display(loan_data[revolving_credit].describe().T)

Variable summary:


,data_type,missing_values
revol_bal,int64,0
revol_util,float64,340
total_rev_hi_lim,float64,70276
tot_cur_bal,float64,70276
tot_coll_amt,float64,70276



Descriptive statistics:


,count,mean,std,min,25%,50%,75%,max
revol_bal,466285.0,16230.203487,20676.245152,0.0,6413.0,11764.0,20333.0,2568995.0
revol_util,465945.0,56.176947,23.732628,0.0,39.2,57.6,74.7,892.3
total_rev_hi_lim,396009.0,30379.087771,37247.129571,0.0,13500.0,22800.0,37900.0,9999999.0
tot_cur_bal,396009.0,138801.713385,152114.663494,0.0,28618.0,81539.0,208953.0,8000078.0
tot_coll_amt,396009.0,191.913517,14630.214605,0.0,0.0,0.0,0.0,9152545.0


## Loan Performance

The variables in this group describe the repayment performance of each loan after it was issued.

They include the outstanding principal, total payments received, recovered amounts, late fees, and other repayment-related measures.

These variables are useful for understanding loan outcomes but may contain information that was not available at the time of the credit decision.

The objective of this assessment is to examine their completeness and descriptive statistics before evaluating potential data leakage.

**Business type:** Numerical

In [ ]:
# Loan performance assessment

loan_performance = [
    "out_prncp",
    "out_prncp_inv",
    "total_pymnt",
    "total_pymnt_inv",
    "total_rec_prncp",
    "total_rec_int",
    "total_rec_late_fee",
    "recoveries",
    "collection_recovery_fee",
    "last_pymnt_amnt"
]

loan_performance_summary = pd.DataFrame({
    "data_type": loan_data[loan_performance].dtypes.astype(str),
    "missing_values": loan_data[loan_performance].isna().sum()
})

print("Variable summary:")
display(loan_performance_summary)

print("\nDescriptive statistics:")
display(loan_data[loan_performance].describe().T)

Variable summary:


,data_type,missing_values
out_prncp,float64,0
out_prncp_inv,float64,0
total_pymnt,float64,0
total_pymnt_inv,float64,0
total_rec_prncp,float64,0
total_rec_int,float64,0
total_rec_late_fee,float64,0
recoveries,float64,0
collection_recovery_fee,float64,0
last_pymnt_amnt,float64,0



Descriptive statistics:


,count,mean,std,min,25%,50%,75%,max
out_prncp,466285.0,4410.062342,6355.078769,0.0,0.000000,441.470000,7341.65000,32160.38000
out_prncp_inv,466285.0,4408.452258,6353.198001,0.0,0.000000,441.380000,7338.39000,32160.38000
total_pymnt,466285.0,11540.686220,8265.627112,0.0,5552.125349,9419.250943,15308.15846,57777.57987
total_pymnt_inv,466285.0,11469.892747,8254.157579,0.0,5499.250000,9355.430000,15231.31000,57777.58000
total_rec_prncp,466285.0,8866.014657,7031.687997,0.0,3708.560000,6817.760000,12000.00000,35000.03000
total_rec_int,466285.0,2588.677225,2483.809661,0.0,957.280000,1818.880000,3304.53000,24205.62000
total_rec_late_fee,466285.0,0.650129,5.265730,0.0,0.000000,0.000000,0.00000,358.68000
recoveries,466285.0,85.344211,552.216084,0.0,0.000000,0.000000,0.00000,33520.27000
collection_recovery_fee,466285.0,8.961534,85.491437,0.0,0.000000,0.000000,0.00000,7002.19000
last_pymnt_amnt,466285.0,3123.913796,5554.737393,0.0,312.620000,545.960000,3187.51000,36234.44000


## Date Variables

The date variables describe important events in the loan lifecycle, including loan issuance, the borrower's earliest credit line, payment activity, and credit monitoring.

These variables were converted to the appropriate datetime format in Section 4. The objective of this assessment is to verify their data types and completeness before feature engineering.

**Business type:** Date/Time

**Decision on Date Variables**

The date variables capture important temporal information describing the loan lifecycle, including loan origination (`issue_d`), the borrower's credit history (`earliest_cr_line`), payment activity (`last_pymnt_d` and `next_pymnt_d`), and the most recent credit record update (`last_credit_pull_d`).

The assessment indicates that all date variables were successfully converted to the appropriate `datetime` data type. The observed date ranges are consistent with the Lending Club loan portfolio and do not reveal any obvious data quality issues. While `issue_d`, `earliest_cr_line`, `last_pymnt_d`, and `last_credit_pull_d` contain very few missing values, `next_pymnt_d` exhibits a large proportion of missing observations. This pattern is expected because many loans have already reached a final status (e.g., Fully Paid or Charged Off), making the next scheduled payment date unavailable.

At this stage, all date variables will be retained in the master dataset. Their predictive value will be assessed during feature engineering, where they may be transformed into more informative numerical features such as credit history length, loan age, time since the last payment, or time since the last credit pull.

**Decision:** Retain all date variables in the master dataset. No additional transformation or imputation is performed in this notebook beyond the conversion to the `datetime` data type.

In [ ]:
# Date variables assessment

date_variables = [
    "issue_d",
    "earliest_cr_line",
    "last_pymnt_d",
    "next_pymnt_d",
    "last_credit_pull_d"
]

date_summary = pd.DataFrame({
    "data_type": loan_data[date_variables].dtypes.astype(str),
    "missing_values": loan_data[date_variables].isna().sum(),
    "minimum_date": loan_data[date_variables].min(),
    "maximum_date": loan_data[date_variables].max()
})

print("Variable summary:")
display(date_summary)

Variable summary:


,data_type,missing_values,minimum_date,maximum_date
issue_d,datetime64[ns],0,2007-06-01,2014-12-01
earliest_cr_line,datetime64[ns],29,1944-01-01,2011-11-01
last_pymnt_d,datetime64[ns],376,2007-12-01,2016-01-01
next_pymnt_d,datetime64[ns],227214,2007-12-01,2016-03-01
last_credit_pull_d,datetime64[ns],42,2007-05-01,2016-01-01


## Administrative Variables

The variables in this group describe administrative or system-related information associated with the loan.

Although these variables may not directly reflect the borrower's financial profile, they may provide useful contextual information or require special treatment during preprocessing.

The objective of this assessment is to examine their completeness and descriptive statistics.

**Business type:** Numerical

In [ ]:
# Administrative variables assessment

administrative_variables = [
    "policy_code",
    "collections_12_mths_ex_med",
    "mths_since_last_delinq",
    "mths_since_last_record",
    "mths_since_last_major_derog"
]

administrative_summary = pd.DataFrame({
    "data_type": loan_data[administrative_variables].dtypes.astype(str),
    "missing_values": loan_data[administrative_variables].isna().sum()
})

print("Variable summary:")
display(administrative_summary)

print("\nDescriptive statistics:")
display(loan_data[administrative_variables].describe().T)

Variable summary:


,data_type,missing_values
policy_code,int64,0
collections_12_mths_ex_med,float64,145
mths_since_last_delinq,float64,250351
mths_since_last_record,float64,403647
mths_since_last_major_derog,float64,367311



Descriptive statistics:


,count,mean,std,min,25%,50%,75%,max
policy_code,466285.0,1.000000,0.000000,1.0,1.0,1.0,1.0,1.0
collections_12_mths_ex_med,466140.0,0.009085,0.108648,0.0,0.0,0.0,0.0,20.0
mths_since_last_delinq,215934.0,34.104430,21.778487,0.0,16.0,31.0,49.0,188.0
mths_since_last_record,62638.0,74.306012,30.357653,0.0,53.0,76.0,102.0,129.0
mths_since_last_major_derog,98974.0,42.852547,21.662591,0.0,26.0,42.0,59.0,188.0


**Decision on Credit History and Delinquency Variables**

The variables `policy_code`, `collections_12_mths_ex_med`, `mths_since_last_delinq`, `mths_since_last_record`, and `mths_since_last_major_derog` describe the borrower's recent credit events and delinquency history. These variables provide information about collections, past delinquencies, public records, and major derogatory events, all of which are potentially relevant predictors of credit risk.

The assessment indicates that `policy_code` is a constant variable, with all observations equal to **1**, and therefore provides no discriminatory information. Consequently, it will be removed from the master dataset.

The remaining variables exhibit varying levels of missingness. While `collections_12_mths_ex_med` contains very few missing values, the three "months since" variables have a substantial proportion of missing observations. However, these missing values are expected from a business perspective, as they generally indicate that the corresponding adverse credit event has never occurred rather than reflecting poor data quality.

At this stage, these variables will be retained in the master dataset because they may contain valuable predictive information. Their missing-value patterns and potential feature engineering strategies will be evaluated during the data preprocessing stage.

**Decision:** Remove `policy_code` because it is a constant feature. Retain `collections_12_mths_ex_med`, `mths_since_last_delinq`, `mths_since_last_record`, and `mths_since_last_major_derog` for subsequent preprocessing and predictive modeling.

# Missing Values Assessment

Missing values are common in real-world financial datasets and may arise for different business reasons. Their presence does not necessarily indicate poor data quality, particularly in credit risk datasets where the absence of information may itself carry meaningful business information.

The objective of this section is to assess the overall missing-value patterns across the dataset, evaluate their potential impact on subsequent analyses, and define an appropriate preprocessing strategy while preserving the original information contained in the data.

Throughout the previous sections, missing values have been examined as part of the assessment of each variable group. This analysis shows that different missing-value mechanisms coexist within the dataset. Some variables contain only a negligible proportion of missing observations, while others exhibit substantial missingness that is consistent with the underlying business processes. For example, variables describing previous delinquencies, public records, or future payment schedules are naturally unavailable for borrowers who have never experienced these events. Consequently, their missing values should not automatically be interpreted as data quality issues.

Only variables containing 100% missing values were removed during the initial data quality assessment. All remaining variables have been retained because they either contain potentially valuable predictive information or require further evaluation during data preprocessing.

At this stage, no missing values are modified. The master dataset preserves the original observations to ensure full traceability and reproducibility throughout the data preparation pipeline. Missing-value treatment—including imputation, creation of missing indicators, or removal of variables if justified—will be performed during the dedicated data preprocessing stage after feature engineering and before predictive modeling.

**Decision:** Preserve the original missing values in the master dataset. No imputation or missing-value treatment is performed in this notebook.

In [ ]:
# Missing values overview

missing_summary = pd.DataFrame({
    "missing_values": loan_data.isna().sum(),
    "missing_percentage": (
        loan_data.isna().mean() * 100
    ).round(2)
})

missing_summary = (
    missing_summary[missing_summary["missing_values"] > 0]
    .sort_values("missing_percentage", ascending=False)
)

display(missing_summary)

,missing_values,missing_percentage
mths_since_last_record,403647,86.57
mths_since_last_major_derog,367311,78.77
desc,340304,72.98
mths_since_last_delinq,250351,53.69
next_pymnt_d,227214,48.73
tot_cur_bal,70276,15.07
tot_coll_amt,70276,15.07
total_rev_hi_lim,70276,15.07
emp_title,27588,5.92
emp_length,21008,4.51


# Feature Engineering

Feature engineering transforms selected raw variables into meaningful and structured features that improve interpretability and support the development of robust credit risk models.

At this stage, only deterministic transformations based on business knowledge are applied. These transformations are independent of the training data and can therefore be shared across the Probability of Default (PD), Loss Given Default (LGD), and Exposure at Default (EAD) modeling datasets.

Model-specific preprocessing steps, statistical imputations, missing-value treatments, target definitions, and data partitioning will be performed later during the data preprocessing phase.



## Employment Length Numerical Transformation

The `emp_length` variable records the borrower's employment duration using ordered textual categories.

Since these categories correspond directly to a measurable quantity, they are converted into their numerical equivalent expressed in years while preserving their natural ordering. For example, `< 1 year` is converted to `0`, while `10+ years` is converted to `10`.

Missing values represent an unknown employment duration rather than a valid employment category. Therefore, they are preserved in the numerical feature and identified separately through the binary indicator `emp_length_missing`.

The creation of `emp_length_missing` is a deterministic feature engineering step because it depends solely on whether the original value is missing and does not require any information from the training data. In contrast, the statistical treatment of missing values in `emp_length_years` (e.g., imputation) will be performed later during the data preprocessing phase using parameters estimated exclusively from the training dataset.

Together, `emp_length_years` and `emp_length_missing` preserve both the borrower's employment duration and the information that the original value was unavailable, allowing subsequent models to exploit both sources of information.

In [ ]:
# Create a missing-value indicator for employment duration

loan_data["emp_length_missing"] = (
    loan_data["emp_length"]
    .isna()
    .astype("Int8")
)

# Convert employment length categories into numerical years

employment_length_mapping = {
    "< 1 year": 0,
    "1 year": 1,
    "2 years": 2,
    "3 years": 3,
    "4 years": 4,
    "5 years": 5,
    "6 years": 6,
    "7 years": 7,
    "8 years": 8,
    "9 years": 9,
    "10+ years": 10
}

loan_data["emp_length_years"] = (
    loan_data["emp_length"]
    .map(employment_length_mapping)
    .astype("Int8")
)

In [ ]:
employment_length_summary = pd.DataFrame({
    "Original employment length": loan_data["emp_length"],
    "Employment length (years)": loan_data["emp_length_years"],
    "Missing indicator": loan_data["emp_length_missing"]
})

display(
    employment_length_summary
    .drop_duplicates()
    .sort_values(
        by=["Missing indicator", "Employment length (years)"],
        na_position="last"
    )
)

,Original employment length,Employment length (years),Missing indicator
1,< 1 year,0,0
4,1 year,1,0
25,2 years,2,0
5,3 years,3,0
8,4 years,4,0
10,5 years,5,0
19,6 years,6,0
30,7 years,7,0
6,8 years,8,0
7,9 years,9,0


## Date Feature Engineering

Several variables describing the loan lifecycle and the borrower's credit history have already been cleaned and converted into the appropriate datetime format during the data preparation stage.

The objective of this section is therefore not to perform additional cleaning, but to derive business-oriented features from these processed dates. These engineered variables provide more informative numerical representations of temporal information while remaining applicable to the PD, LGD, and EAD datasets.

**Credit History Length**

The borrower's credit history is an important indicator of creditworthiness.

Rather than using the original account opening date (`earliest_cr_line`), the duration of the borrower's credit history at the loan issuance date is computed. A new numerical feature is created:

- `credit_history_months`: the length of the borrower's credit history expressed in months;

These engineered features provide a more informative representation of the applicant's credit experience than the original date variable while remaining directly usable in subsequent credit risk models.

In [ ]:
# Create credit history feature

loan_data["credit_history_months"] = (
    (
        loan_data["issue_d"].dt.year
        - loan_data["earliest_cr_line"].dt.year
    ) * 12
    + (
        loan_data["issue_d"].dt.month
        - loan_data["earliest_cr_line"].dt.month
    )
).astype("Int16")


In [ ]:
# Validate engineered feature

credit_history_validation = loan_data[
    [
        "earliest_cr_line",
        "issue_d",
        "credit_history_months"
    ]
]

display(credit_history_validation.head())

print(
    f"Negative values: "
    f"{(loan_data['credit_history_months'] < 0).sum():,}"
)

print(
    f"Missing values: "
    f"{loan_data['credit_history_months'].isna().sum():,}"
)

,earliest_cr_line,issue_d,credit_history_months
0,1985-01-01,2011-12-01,323
1,1999-04-01,2011-12-01,152
2,2001-11-01,2011-12-01,121
3,1996-02-01,2011-12-01,190
4,1996-01-01,2011-12-01,191


Negative values: 0
Missing values: 29


## Credit Grade Features

The variables `grade` and `sub_grade` are ordinal categorical variables representing Lending Club's internal assessment of borrower credit risk.

Based on the analysis performed in Section 5, both variables are converted from `object` to ordered categorical types while preserving their natural risk hierarchy. No numerical encoding or missing-value treatment is applied at this stage. The final decision regarding redundancy between `grade` and `sub_grade` will be made during Feature Selection.

In [ ]:
# Automatically extract and order the categories
grade_order = sorted(
    loan_data["grade"].dropna().unique()
)

sub_grade_order = sorted(
    loan_data["sub_grade"].dropna().unique(),
    key=lambda value: (value[0], int(value[1:]))
)

# Convert to ordered categorical variables
loan_data["grade"] = pd.Categorical(
    loan_data["grade"],
    categories=grade_order,
    ordered=True
)

loan_data["sub_grade"] = pd.Categorical(
    loan_data["sub_grade"],
    categories=sub_grade_order,
    ordered=True
)

In [ ]:
for variable in ["grade", "sub_grade"]:
    print(f"\n{variable}")
    print("-" * 30)
    print("Data type:", loan_data[variable].dtype)
    print("Ordered:", loan_data[variable].cat.ordered)
    print("Missing values:", loan_data[variable].isna().sum())
    print("Categories:", loan_data[variable].cat.categories.tolist())


grade
------------------------------
Data type: category
Ordered: True
Missing values: 0
Categories: ['A', 'B', 'C', 'D', 'E', 'F', 'G']

sub_grade
------------------------------
Data type: category
Ordered: True
Missing values: 0
Categories: ['A1', 'A2', 'A3', 'A4', 'A5', 'B1', 'B2', 'B3', 'B4', 'B5', 'C1', 'C2', 'C3', 'C4', 'C5', 'D1', 'D2', 'D3', 'D4', 'D5', 'E1', 'E2', 'E3', 'E4', 'E5', 'F1', 'F2', 'F3', 'F4', 'F5', 'G1', 'G2', 'G3', 'G4', 'G5']


In [ ]:
print((loan_data["loan_amnt"] == loan_data["funded_amnt"]).mean())

print((loan_data["funded_amnt"] == loan_data["funded_amnt_inv"]).mean())

0.9955778118532657
0.8525922987014379


## Loan Amount Features

The Lending Club dataset contains three variables describing different stages of the loan funding process: `loan_amnt`, `funded_amnt`, and `funded_amnt_inv`. To better capture these relationships, three business-oriented features are created.

- **`funding_ratio`** measures the proportion of the requested loan amount that was approved and funded. A value of 1 indicates that the borrower received the full requested amount, while lower values indicate partial funding.

- **`investor_funding_ratio`** measures the proportion of the funded loan amount that was financed by investors. This feature reflects the level of investor participation after the loan has been approved.

- **`investor_loan_ratio`** measures the proportion of the originally requested loan amount that was ultimately financed by investors. It combines information about both the approval process and investor participation.

These normalized ratios provide more comparable measures than absolute loan amounts across borrowers requesting different loan sizes.

Since `loan_amnt`, `funded_amnt`, and `funded_amnt_inv` contain no missing values, these engineered features can be computed directly without any additional preprocessing.

In [ ]:
# Percentage of the requested loan that was funded
loan_data["funding_ratio"] = (
    loan_data["funded_amnt"] / loan_data["loan_amnt"]
).astype("float32")

# Percentage of the funded loan provided by investors
loan_data["investor_funding_ratio"] = (
    loan_data["funded_amnt_inv"] / loan_data["funded_amnt"]
).astype("float32")

# Percentage of the requested loan funded by investors
loan_data["investor_loan_ratio"] = (
    loan_data["funded_amnt_inv"] / loan_data["loan_amnt"]
).astype("float32")

In [ ]:
loan_amount_features = [
    "funding_ratio",
    "investor_funding_ratio",
    "investor_loan_ratio"
]

loan_data[loan_amount_features].describe()

,funding_ratio,investor_funding_ratio,investor_loan_ratio
count,466285.000000,466285.000000,466285.000000
mean,0.998641,0.993578,0.992447
std,0.022070,0.060647,0.064484
min,0.101250,0.000000,0.000000
25%,1.000000,1.000000,1.000000
50%,1.000000,1.000000,1.000000
75%,1.000000,1.000000,1.000000
max,1.000000,1.000000,1.000000


## Income and Repayment Capacity Features

The variables `annual_inc`, `loan_amnt`, `installment`, and `dti` describe the borrower's financial capacity and the repayment burden associated with the loan. Together, they provide complementary information about the borrower's ability to repay the requested credit.

To better quantify repayment capacity, two normalized financial ratios are created.

- **`loan_to_income_ratio`** measures the proportion of the borrower's annual income represented by the requested loan amount. Higher values indicate a larger financial commitment relative to income and may be associated with higher credit risk.

- **`installment_to_income_ratio`** measures the proportion of the borrower's annual income required to cover the annual loan repayments. Since `installment` is reported as a monthly payment while `annual_inc` represents annual income, the monthly installment is multiplied by 12 before computing the ratio.

The dataset already includes the variable `dti` (Debt-to-Income Ratio), which measures the borrower's existing debt burden relative to income, excluding the requested Lending Club loan. Since this variable already captures debt affordability, no additional debt-to-income feature is engineered to avoid redundancy.

The engineered variables complement the existing financial indicators by quantifying the relative size of the requested loan and its repayment burden with respect to the borrower's income. Their predictive contribution will be evaluated during the feature selection stage.

In [ ]:
# Create a copy to avoid division by zero
income = loan_data["annual_inc"].replace(0, np.nan)

# Loan amount relative to annual income
loan_data["loan_to_income_ratio"] = (
    loan_data["loan_amnt"] / income
)

# Annual installment relative to annual income
loan_data["installment_to_income_ratio"] = (
    loan_data["installment"] * 12 / income
)

In [ ]:
#Validation
loan_data[
    [
        "annual_inc",
        "loan_to_income_ratio",
        "installment_to_income_ratio"
    ]
].isnull().sum()

annual_inc                     4
loan_to_income_ratio           4
installment_to_income_ratio    4
dtype: int64

In [ ]:
# Engineered repayment capacity features
repayment_features = [
    "loan_to_income_ratio",
    "installment_to_income_ratio"
]

# Summary statistics
display(loan_data[repayment_features].describe())

,loan_to_income_ratio,installment_to_income_ratio
count,466281.000000,466281.000000
mean,0.218683,0.080102
std,0.110894,0.040276
min,0.000789,0.000289
25%,0.131579,0.049087
50%,0.205714,0.075190
75%,0.298507,0.107158
max,1.337500,0.541710


## Credit History Features

A borrower's previous credit behavior is one of the strongest predictors of future credit performance. While the dataset already contains several variables describing the applicant's credit history, additional normalized features can better summarize the borrower's financial behavior and provide informative inputs for credit risk modeling.

Four business-oriented variables are created.

- **`open_account_ratio`** measures the proportion of currently active credit accounts relative to the total number of credit accounts. Higher values indicate that a larger share of the borrower's credit history remains active.

- **`credit_inquiry_rate`** measures the number of recent credit inquiries relative to the length of the borrower's credit history (`credit_history_months`). This normalizes recent credit-seeking behavior by the borrower's credit experience.

- **`delinquency_rate`** measures the proportion of delinquent accounts relative to the total number of credit accounts. It summarizes the frequency of recent payment problems.


These engineered variables summarize complementary aspects of borrowers' credit behavior while remaining interpretable and comparable across individuals with different credit histories. They provide meaningful candidate predictors for the subsequent PD, LGD, and EAD models, while their final predictive contribution will be evaluated during the feature selection stage.

In [ ]:
# Ratio of currently open accounts
loan_data["open_account_ratio"] = (
    loan_data["open_acc"] /
    loan_data["total_acc"]
)

# Credit inquiries normalized by credit history (years)
loan_data["credit_inquiry_rate"] = (
    loan_data["inq_last_6mths"] /
    (loan_data["credit_history_months"] / 12)
)

# Delinquency frequency
loan_data["delinquency_rate"] = (
    loan_data["delinq_2yrs"] /
    loan_data["total_acc"]
)


In [ ]:
new_features = [
    "open_account_ratio",
    "credit_inquiry_rate",
    "delinquency_rate"
]

loan_data[new_features].describe().T

,count,mean,std,min,25%,50%,75%,max
open_account_ratio,466256.0,0.484942,0.172545,0.0,0.358974,0.464286,0.590909,1.75
credit_inquiry_rate,466256.0,0.064161,0.127777,0.0,0.0,0.0,0.090909,22.5
delinquency_rate,466256.0,0.011541,0.032706,0.0,0.0,0.0,0.0,1.0


In [ ]:
loan_data.loc[
    loan_data["open_account_ratio"] > 1,
    ["open_acc", "total_acc", "open_account_ratio"]
].head(20)

,open_acc,total_acc,open_account_ratio
39711,14.0,8.0,1.750000
42432,4.0,3.0,1.333333


**Note**

A small number of observations exhibit `open_account_ratio` values greater than 1 because the reported number of open credit accounts exceeds the reported total number of credit accounts in the original dataset. In addition, a few observations have `public_record_rate` values greater than 1, reflecting borrowers with more derogatory public records than reported credit accounts. Since these values originate from the original Lending Club dataset, they are retained without modification to preserve data integrity.

## Borrower Profile Features

Several variables describe the borrower's personal and employment profile at the time of the loan application, including home ownership, employment characteristics, income verification status, and application type.

Most of these variables already provide meaningful business information and therefore require only limited feature engineering. The objective of this section is to improve data consistency, preserve potentially informative missingness patterns, and remove variables that provide no predictive value.

The following transformations are applied:

- The rare home ownership categories (`ANY` and `NONE`) are grouped into the `OTHER` category to reduce sparsity while preserving business meaning.
- `home_ownership`, `verification_status`, and `emp_length` are converted to the `category` data type to better reflect their categorical nature.
- `emp_title_missing` is created to indicate whether the borrower's employment title is missing, since the absence of this information may itself be informative for subsequent predictive models.
- `application_type` is removed because it contains a single category (`INDIVIDUAL`) and therefore provides no discriminatory information.

In [ ]:
# Group very rare home ownership categories
loan_data["home_ownership"] = loan_data["home_ownership"].replace(
    {
        "ANY": "OTHER",
        "NONE": "OTHER"
    }
)

# Convert borrower profile variables to categorical type
loan_data["home_ownership"] = loan_data["home_ownership"].astype("category")
loan_data["verification_status"] = loan_data["verification_status"].astype("category")
loan_data["emp_length"] = loan_data["emp_length"].astype("category")

# Create missing-value indicator for employment title
loan_data["emp_title_missing"] = (
    loan_data["emp_title"]
    .isna()
    .astype("int8")
)

# Remove constant feature
loan_data.drop(columns=["application_type"], inplace=True)

In [ ]:
loan_data["home_ownership"].value_counts()

home_ownership
MORTGAGE    235875
RENT        188473
OWN          41704
OTHER          233
Name: count, dtype: int64

## Interaction Features

Certain borrower characteristics may jointly describe credit risk more effectively than when considered independently.

A business-driven interaction feature is created by combining the borrower's repayment burden with the interest rate assigned to the loan.

The resulting variable, `loan_burden_interest`, identifies borrowers who simultaneously face a high repayment burden relative to their income and a high borrowing cost. Such borrowers may represent a higher credit risk than suggested by either factor alone.

Because this interaction is computed exclusively from variables available at the time of loan origination, it can be safely included in the Master Credit Risk Dataset without introducing target leakage.

In [ ]:
loan_data["loan_burden_interest"] = (
    loan_data["loan_to_income_ratio"] *
    loan_data["int_rate"]
)


## Missing Value Handling and Variable Removal

This section handles variables with informative missing values and removes variables that are not suitable for the Master Credit Risk Dataset.

For selected time-related variables, missing values may contain useful business information. Therefore, two complementary transformations are applied:

- a binary missing indicator (`*_is_missing`) is created to preserve the information that the original value was missing;
- missing numerical values are imputed with **`-1`**, a sentinel value that cannot naturally occur for variables representing elapsed time in months.

The value **`-1` does not represent one month before the event**. Instead, it is an artificial placeholder used solely to distinguish originally missing observations from valid elapsed-time values (which are always non-negative). The corresponding missing indicator allows machine learning models to identify these observations explicitly.

Finally, variables that are redundant, administrative, non-informative, or unsuitable for predictive modeling are removed.

Specifically:

- `url` is removed because it only contains a Lending Club webpage link.
- `desc` is removed because it is an unstructured text field that will not be used for modeling.
- `application_type` is removed because it is constant in the available dataset.
- `policy_code` is removed because it is an administrative variable with no useful borrower-risk information.
- `pymnt_plan` is removed because it provides little or no predictive information at loan origination.

In [ ]:
# ---------------------------------------------------------------------
# Missing Value Handling
# ---------------------------------------------------------------------

missing_time_variables = [
    "mths_since_last_record",
    "mths_since_last_major_derog",
    "mths_since_last_delinq"
]

for column in missing_time_variables:

    # Preserve the information that the original value was missing
    loan_data[f"{column}_is_missing"] = (
        loan_data[column]
        .isna()
        .astype("int8")
    )

    # Replace missing values with a sentinel value (-1)
    loan_data[column] = (
        loan_data[column]
        .fillna(-1)
    )

print("Missing value handling completed.")

# ---------------------------------------------------------------------
# Variable Removal
# ---------------------------------------------------------------------

variables_to_remove = [
    "url",
    "desc",
    "application_type",
    "policy_code",
    "pymnt_plan"
]

removed_variables = [
    variable
    for variable in variables_to_remove
    if variable in loan_data.columns
]

loan_data.drop(
    columns=variables_to_remove,
    inplace=True,
    errors="ignore"
)

print("\nRemoved variables:")
for variable in removed_variables:
    print(f" - {variable}")

print(
    f"\nMaster Dataset dimensions: "
    f"{loan_data.shape[0]:,} rows × {loan_data.shape[1]:,} columns"
)

# Verify that no missing values remain
loan_data[
    [
        "mths_since_last_record",
        "mths_since_last_major_derog",
        "mths_since_last_delinq"
    ]
].isna().sum()

Missing value handling completed.

Removed variables:
 - url
 - desc
 - policy_code
 - pymnt_plan

Master Dataset dimensions: 466,285 rows × 68 columns


mths_since_last_record         0
mths_since_last_major_derog    0
mths_since_last_delinq         0
dtype: int64

In [ ]:
loan_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 466285 entries, 0 to 466284
Data columns (total 68 columns):
 #   Column                                  Non-Null Count   Dtype         
---  ------                                  --------------   -----         
 0   id                                      466285 non-null  int64         
 1   member_id                               466285 non-null  int64         
 2   loan_amnt                               466285 non-null  int64         
 3   funded_amnt                             466285 non-null  int64         
 4   funded_amnt_inv                         466285 non-null  float64       
 5   term                                    466285 non-null  int16         
 6   int_rate                                466285 non-null  float64       
 7   installment                             466285 non-null  float64       
 8   grade                                   466285 non-null  category      
 9   sub_grade                            

In [ ]:
print("=" * 60)
print("MASTER DATASET SUMMARY")
print("=" * 60)

print(f"Rows: {loan_data.shape[0]:,}")
print(f"Columns: {loan_data.shape[1]}")

print("\nMemory usage:")
print(f"{loan_data.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

print("\nMissing values remaining:")
print(
    loan_data.isna()
             .sum()
             .loc[lambda x: x > 0]
             .sort_values(ascending=False)
)

MASTER DATASET SUMMARY
Rows: 466,285
Columns: 68

Memory usage:
354.90 MB

Missing values remaining:
next_pymnt_d                   227214
total_rev_hi_lim                70276
tot_cur_bal                     70276
tot_coll_amt                    70276
emp_title                       27588
emp_length                      21008
emp_length_years                21008
last_pymnt_d                      376
revol_util                        340
collections_12_mths_ex_med        145
last_credit_pull_d                 42
acc_now_delinq                     29
credit_history_months              29
open_acc                           29
total_acc                          29
earliest_cr_line                   29
inq_last_6mths                     29
delinq_2yrs                        29
pub_rec                            29
delinquency_rate                   29
open_account_ratio                 29
credit_inquiry_rate                29
title                              21
annual_inc               

# Final Validation and Export

This section performs the final quality checks on the Master Credit Risk Dataset before export.

The validation covers:

- dataset dimensions and duplicate rows;
- duplicate column names;
- remaining missing values;
- infinite values generated by ratio calculations;
- consistency of engineered ratio variables;
- target-status availability for downstream PD modelling;
- memory usage and data types;
- export of the cleaned Master Dataset and supporting quality reports.

The Master Dataset intentionally preserves variables required for later PD, LGD and EAD dataset construction. Model-specific transformations such as train/test splitting, binning, WoE, IV, statistical imputation and feature selection are not performed here.

In [ ]:
# ---------------------------------------------------------------------
# Final structural validation
# ---------------------------------------------------------------------

print("=" * 70)
print("FINAL MASTER DATASET VALIDATION")
print("=" * 70)

row_count, column_count = loan_data.shape
duplicate_rows = int(loan_data.duplicated().sum())
duplicate_columns = loan_data.columns[loan_data.columns.duplicated()].tolist()

print(f"Rows: {row_count:,}")
print(f"Columns: {column_count:,}")
print(f"Duplicate rows: {duplicate_rows:,}")
print(f"Duplicate column names: {len(duplicate_columns):,}")

if duplicate_columns:
    print("Duplicate columns detected:")
    for column in duplicate_columns:
        print(f" - {column}")

assert row_count > 0, "The Master Dataset is empty."
assert column_count > 0, "The Master Dataset contains no columns."
assert not duplicate_columns, "Duplicate column names must be resolved before export."

print("\nStructural validation passed.")

FINAL MASTER DATASET VALIDATION


Rows: 466,285
Columns: 68
Duplicate rows: 0
Duplicate column names: 0

Structural validation passed.


In [ ]:
# ---------------------------------------------------------------------
# Missing-value and cardinality report
# ---------------------------------------------------------------------

quality_report = pd.DataFrame({
    "dtype": loan_data.dtypes.astype(str),
    "missing_values": loan_data.isna().sum(),
    "missing_percentage": loan_data.isna().mean().mul(100),
    "unique_values": loan_data.nunique(dropna=False)
})

quality_report = quality_report.sort_values(
    by=["missing_percentage", "unique_values"],
    ascending=[False, False]
)

print("Columns with remaining missing values:")
display(
    quality_report.loc[quality_report["missing_values"] > 0]
)

print("\nColumns with one unique value:")
display(
    quality_report.loc[quality_report["unique_values"] <= 1]
)

Columns with remaining missing values:


,dtype,missing_values,missing_percentage,unique_values
next_pymnt_d,datetime64[ns],227214,48.728567,101
tot_cur_bal,float64,70276,15.071469,220691
total_rev_hi_lim,float64,70276,15.071469,14613
tot_coll_amt,float64,70276,15.071469,6322
emp_title,object,27588,5.916553,205476
emp_length,category,21008,4.505399,12
emp_length_years,Int8,21008,4.505399,12
last_pymnt_d,datetime64[ns],376,0.080637,99
revol_util,float64,340,0.072917,1270
collections_12_mths_ex_med,float64,145,0.031097,10



Columns with one unique value:


,dtype,missing_values,missing_percentage,unique_values


In [ ]:
# ---------------------------------------------------------------------
# Infinite-value validation for numeric and engineered variables
# ---------------------------------------------------------------------

numeric_columns = loan_data.select_dtypes(include=[np.number]).columns

infinite_report = pd.DataFrame({
    "positive_infinity": [
        np.isposinf(loan_data[column]).sum()
        for column in numeric_columns
    ],
    "negative_infinity": [
        np.isneginf(loan_data[column]).sum()
        for column in numeric_columns
    ]
}, index=numeric_columns)

infinite_report["total_infinite"] = (
    infinite_report["positive_infinity"]
    + infinite_report["negative_infinity"]
)

infinite_report = infinite_report.loc[
    infinite_report["total_infinite"] > 0
].sort_values("total_infinite", ascending=False)

if infinite_report.empty:
    print("No infinite values detected.")
else:
    print("Infinite values detected:")
    display(infinite_report)

    affected_columns = infinite_report.index.tolist()
    loan_data[affected_columns] = loan_data[affected_columns].replace(
        [np.inf, -np.inf],
        np.nan
    )

    print(
        "Infinite values were replaced with NaN so they remain visible "
        "for downstream model-specific treatment."
    )

No infinite values detected.


In [ ]:
# ---------------------------------------------------------------------
# Engineered-feature consistency checks
# ---------------------------------------------------------------------

consistency_checks = {}

if {"open_acc", "total_acc"}.issubset(loan_data.columns):
    consistency_checks["open_acc_greater_than_total_acc"] = int(
        (loan_data["open_acc"] > loan_data["total_acc"]).sum()
    )

if "loan_to_income_ratio" in loan_data.columns:
    consistency_checks["negative_loan_to_income_ratio"] = int(
        (loan_data["loan_to_income_ratio"] < 0).sum()
    )

if "installment_to_income_ratio" in loan_data.columns:
    consistency_checks["negative_installment_to_income_ratio"] = int(
        (loan_data["installment_to_income_ratio"] < 0).sum()
    )

if "credit_history_months" in loan_data.columns:
    consistency_checks["negative_credit_history_months"] = int(
        (loan_data["credit_history_months"] < 0).sum()
    )

consistency_report = pd.DataFrame.from_dict(
    consistency_checks,
    orient="index",
    columns=["record_count"]
)

display(consistency_report)

if (
    "open_acc_greater_than_total_acc" in consistency_checks
    and consistency_checks["open_acc_greater_than_total_acc"] > 0
):
    loan_data["open_account_inconsistency"] = (
        loan_data["open_acc"] > loan_data["total_acc"]
    ).astype("int8")

    print(
        "Created 'open_account_inconsistency' to preserve the detected "
        "source-data inconsistency without altering the original fields."
    )

,record_count
open_acc_greater_than_total_acc,2
negative_loan_to_income_ratio,0
negative_installment_to_income_ratio,0
negative_credit_history_months,0


Created 'open_account_inconsistency' to preserve the detected source-data inconsistency without altering the original fields.


In [ ]:
# ---------------------------------------------------------------------
# Downstream modelling readiness checks
# ---------------------------------------------------------------------

required_pd_columns = ["loan_status"]
missing_required_columns = [
    column
    for column in required_pd_columns
    if column not in loan_data.columns
]

assert not missing_required_columns, (
    "Required downstream columns are missing: "
    f"{missing_required_columns}"
)

print("Loan-status distribution retained for downstream target definition:")
display(
    loan_data["loan_status"]
    .value_counts(dropna=False)
    .rename_axis("loan_status")
    .to_frame("record_count")
)

print("Downstream PD dataset construction requirements are available.")

Loan-status distribution retained for downstream target definition:


,record_count
loan_status,
Current,224226
Fully Paid,184739
Charged Off,42475
Late (31-120 days),6900
In Grace Period,3146
Does not meet the credit policy. Status:Fully Paid,1988
Late (16-30 days),1218
Default,832
Does not meet the credit policy. Status:Charged Off,761


Downstream PD dataset construction requirements are available.


In [ ]:
# ---------------------------------------------------------------------
# Rebuild final reports after validation corrections
# ---------------------------------------------------------------------

quality_report = pd.DataFrame({
    "dtype": loan_data.dtypes.astype(str),
    "missing_values": loan_data.isna().sum(),
    "missing_percentage": loan_data.isna().mean().mul(100),
    "unique_values": loan_data.nunique(dropna=False)
}).sort_values(
    by=["missing_percentage", "unique_values"],
    ascending=[False, False]
)

dataset_summary = pd.DataFrame({
    "metric": [
        "rows",
        "columns",
        "duplicate_rows",
        "columns_with_missing_values",
        "total_missing_values",
        "numeric_columns",
        "categorical_columns",
        "memory_usage_mb"
    ],
    "value": [
        loan_data.shape[0],
        loan_data.shape[1],
        loan_data.duplicated().sum(),
        (loan_data.isna().sum() > 0).sum(),
        loan_data.isna().sum().sum(),
        loan_data.select_dtypes(include=[np.number]).shape[1],
        loan_data.select_dtypes(exclude=[np.number]).shape[1],
        round(loan_data.memory_usage(deep=True).sum() / 1024**2, 2)
    ]
})

display(dataset_summary)


,metric,value
0,rows,466285.00
1,columns,69.00
2,duplicate_rows,0.00
3,columns_with_missing_values,27.00
4,total_missing_values,508905.00
5,numeric_columns,52.00
6,categorical_columns,17.00
7,memory_usage_mb,355.34


In [ ]:
# ---------------------------------------------------------------------
# Export Master Dataset and quality reports
# ---------------------------------------------------------------------

PROCESSED_DIR = Path("../data/processed")
QUALITY_REPORT_DIR = Path("../reports/data_quality")

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
QUALITY_REPORT_DIR.mkdir(parents=True, exist_ok=True)

master_parquet_path = PROCESSED_DIR / "master_credit_risk_dataset.parquet"
master_csv_path = PROCESSED_DIR / "master_credit_risk_dataset.csv"
quality_report_path = QUALITY_REPORT_DIR / "master_dataset_quality_report.csv"
summary_report_path = QUALITY_REPORT_DIR / "master_dataset_summary.csv"
consistency_report_path = QUALITY_REPORT_DIR / "master_dataset_consistency_report.csv"

# Parquet is the primary analytical format because it preserves data types
# and is more efficient than CSV for downstream processing.
try:
    loan_data.to_parquet(
        master_parquet_path,
        index=False
    )
    parquet_exported = True
except ImportError as error:
    parquet_exported = False
    print(
        "Parquet export skipped because no Parquet engine is installed. "
        "Install pyarrow or fastparquet to enable it."
    )
    print(f"Details: {error}")

# CSV is retained as a portable exchange format.
loan_data.to_csv(master_csv_path, index=False)
quality_report.to_csv(quality_report_path, index=True, index_label="column")
dataset_summary.to_csv(summary_report_path, index=False)
consistency_report.to_csv(
    consistency_report_path,
    index=True,
    index_label="check"
)

print("=" * 70)
print("EXPORT COMPLETED")
print("=" * 70)

if parquet_exported:
    print(f"Master Dataset (Parquet): {master_parquet_path.resolve()}")

print(f"Master Dataset (CSV):     {master_csv_path.resolve()}")
print(f"Quality report:           {quality_report_path.resolve()}")
print(f"Dataset summary:          {summary_report_path.resolve()}")
print(f"Consistency report:       {consistency_report_path.resolve()}")

EXPORT COMPLETED
Master Dataset (Parquet): C:\Users\Platini AGOUANET\OneDrive\Desktop\Mes projets IA\risk-credit-scoring-new\data\processed\master_credit_risk_dataset.parquet
Master Dataset (CSV):     C:\Users\Platini AGOUANET\OneDrive\Desktop\Mes projets IA\risk-credit-scoring-new\data\processed\master_credit_risk_dataset.csv
Quality report:           C:\Users\Platini AGOUANET\OneDrive\Desktop\Mes projets IA\risk-credit-scoring-new\reports\data_quality\master_dataset_quality_report.csv
Dataset summary:          C:\Users\Platini AGOUANET\OneDrive\Desktop\Mes projets IA\risk-credit-scoring-new\reports\data_quality\master_dataset_summary.csv
Consistency report:       C:\Users\Platini AGOUANET\OneDrive\Desktop\Mes projets IA\risk-credit-scoring-new\reports\data_quality\master_dataset_consistency_report.csv


## Master Dataset Completion

The Master Credit Risk Dataset is now structurally validated and exported.

The next notebook will construct the dedicated **Probability of Default (PD) modelling dataset** by:

1. defining the binary target from `loan_status`;
2. retaining only loans with an observable final outcome;
3. excluding variables unavailable at loan origination or likely to cause target leakage;
4. separating predictors and target;
5. preparing the dataset for train/test splitting, binning, Weight of Evidence and Information Value analysis.